[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_04/TSA_ch4_seasonal_data/TSA_ch4_seasonal_data.ipynb)

# TSA_ch4_seasonal_data

Seasonality in Romanian data: real GDP (not adjusted), retail trade, industrial production, tourism nights, monthly HICP inflation (Eurostat) and daily electricity load (ENTSO-E); a seasonal plot of tourism nights and the quarterly subseries of GDP; Eurostat seasonally and calendar adjusted (SCA) against unadjusted (NSA) series of GDP and retail trade, with the implied seasonal factors.

*Time Series Analysis (TSA), Chapter 4: Seasonality and forecasting: SARIMA, TBATS, Prophet. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_4'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import logging

import matplotlib.dates as mdates

from statsmodels.tsa.stattools import acf, pacf

from statsmodels.tsa.statespace.sarimax import SARIMAX

from statsmodels.tsa.arima_process import ArmaProcess

from statsmodels.tsa.holtwinters import ExponentialSmoothing

from statsmodels.tsa.seasonal import MSTL

from statsmodels.regression.linear_model import OLS

from dateutil.easter import easter, EASTER_ORTHODOX

logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

logging.getLogger('prophet').setLevel(logging.WARNING)

SEED = 2026

GDP_NSA = ('namq_10_gdp', 'Q.CLV10_MEUR.NSA.B1GQ.RO')

GDP_SCA = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

RETAIL = ('sts_trtu_m', 'M.VOL_SLS.G47.NSA.I21.RO')

RETAIL_SCA = ('sts_trtu_m', 'M.VOL_SLS.G47.SCA.I21.RO')

FOOD = ('sts_trtu_m', 'M.VOL_SLS.G47_FOOD.NSA.I21.RO')

IPI = ('sts_inpr_m', 'M.PRD.B-D.NSA.I21.RO')

TOUR = ('tour_occ_nim', 'M.TOTAL.NR.I551-I553.RO')

HICP = ('prc_hicp_minr', 'M.I15.TOTAL.RO')

GDP_START = '2000-01-01'

ENTSOE = 'https://www.entsoe.eu/publications/data/power-stats/{y}/monthly_hourly_load_values_{y}.csv'

LOAD_FILE = 'ch4_ro_load_hourly.csv'

LOAD_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/Quantlets/Ch_04/ch4_ro_load_hourly.csv'

LOAD_YEARS = (2022, 2023, 2024, 2025, 2026)

LOAD_END = '2026-06-30'

CV_FIRST, CV_STEP, CV_H = '2025-06-30', 14, 14

BAND_COL = st.IDAred

MODEL_COL = {'Seasonal naive': st.Amber, 'ETS': st.Teal, 'SARIMA': st.Purple, 'DHR': st.MainBlue, 'TBATS': st.Forest, 'Prophet': st.Orange, 'Combination': st.IDAred}

GDP_GRID = [((0, 1, 1), (0, 1, 1)), ((1, 1, 0), (0, 1, 1)), ((1, 1, 1), (0, 1, 1)), ((0, 1, 2), (0, 1, 1)), ((2, 1, 0), (0, 1, 1)), ((0, 1, 1), (1, 1, 0)), ((1, 1, 0), (1, 1, 0)), ((0, 1, 1), (1, 1, 1)), ((0, 1, 0), (0, 1, 1)), ((0, 1, 1), (0, 1, 0))]

HERE = "."

def have(pkg):
    """True if an optional package (tbats, prophet, pmdarima) is installed."""
    import importlib.util
    return importlib.util.find_spec(pkg) is not None


def eurostat_log(key, start=None):
    """Natural log of a Eurostat series (monthly or quarterly), from `start`."""
    s = np.log(read_eurostat(*key))
    s = s.loc[start:] if start else s
    s.index.freq = None
    return s


def ro_holidays(years):
    """Romanian public holidays (Orthodox Easter and Pentecost from dateutil): date -> name."""
    out = {}
    for y in years:
        e = pd.Timestamp(easter(y, EASTER_ORTHODOX))
        day = pd.Timedelta(days=1)
        hol = {'New Year': [f'{y}-01-01', f'{y}-01-02'], 'Union Day': [f'{y}-01-24'],
               'Easter': [e - 2 * day, e, e + day], 'Labour Day': [f'{y}-05-01'], "Children's Day": [f'{y}-06-01'],
               'Pentecost': [e + 49 * day, e + 50 * day], 'Assumption': [f'{y}-08-15'], "St Andrew's Day": [f'{y}-11-30'],
               'National Day': [f'{y}-12-01'], 'Christmas': [f'{y}-12-25', f'{y}-12-26']}
        if y >= 2024:
            hol['Epiphany'] = [f'{y}-01-06', f'{y}-01-07']
        for k, v in hol.items():
            for x in v:
                out[pd.Timestamp(x)] = k
    return pd.Series(out, name='holiday').sort_index()


def load_hourly(years=LOAD_YEARS, end=LOAD_END):
    """Hourly electricity load of Romania (MW, hourly average), ENTSO-E 'monthly hourly load values'.
    Reads the extract ch4_ro_load_hourly.csv (local or from the TSA repository); otherwise downloads the yearly ENTSO-E
    files (about 40 MB each) and keeps Romania. Returns the series in UTC+2 (Romanian winter time, no clock change),
    with isolated one-hour glitches (more than 30% away from both neighbours) and missing hours interpolated."""
    s = None
    local = [os.path.join(HERE, LOAD_FILE)] + [os.path.join(d, 'Quantlets', 'Ch_04', LOAD_FILE) for d in ('.', '..', '../..', '../../..')]
    for src in [p for p in local if os.path.exists(p)] + [LOAD_RAW]:
        try:
            s = pd.read_csv(src, index_col=0, parse_dates=True).iloc[:, 0]
            break
        except Exception:
            continue
    if s is None:
        parts = []
        for y in years:
            url = ENTSOE.format(y=y)
            head = pd.read_csv(url, nrows=2, encoding='utf-8-sig', sep=None, engine='python')
            sep = '\t' if '\t' in ''.join(head.columns) or len(head.columns) == 1 else ','
            t = pd.read_csv(url, sep=sep, usecols=['DateUTC', 'CountryCode', 'Value'], encoding='utf-8-sig')
            t = t[t['CountryCode'] == 'RO']
            parts.append(pd.Series(t['Value'].astype(float).values,
                                   index=pd.to_datetime(t['DateUTC'], format='%d-%m-%Y %H:%M')))
        s = pd.concat(parts)
        s = s[~s.index.duplicated()].sort_index().rename('load_MW')
        s.index.name = 'date_utc'
        try:
            s.to_csv(os.path.join(HERE, LOAD_FILE) if os.path.isdir(HERE) else LOAD_FILE)
        except OSError:
            pass
    s = s.loc[:pd.Timestamp(end) + pd.Timedelta(hours=21)]
    s.index = s.index + pd.Timedelta(hours=2)
    s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq='h'))
    r = s / ((s.shift(1) + s.shift(-1)) / 2)
    s[(r < 0.7) | (r > 1.3)] = np.nan
    return s.interpolate().rename('load')


def load_daily(end=LOAD_END):
    """Daily mean load in GW (Romanian winter time), 1 January 2022 - 30 June 2026."""
    d = load_hourly().resample('D').mean() / 1000
    return d.loc['2022-01-01':end].rename('load')


def sample_acf(x, nlags=20):
    return acf(np.asarray(x, float), nlags=nlags, fft=True)[1:]


def sample_pacf(x, nlags=20):
    return pacf(np.asarray(x, float), nlags=nlags, method='ywm')[1:]


def ljung_box(x, m=24, df=None):
    """Ljung-Box Q*(m) with df degrees of freedom (m minus the number of ARMA parameters for residuals)."""
    x = np.asarray(x, float)
    T = len(x)
    r = sample_acf(x, m)
    q = T * (T + 2) * np.sum(r ** 2 / (T - np.arange(1, m + 1)))
    df = m if df is None else df
    return {'lb': float(q), 'df': int(df), 'lb_p': float(stats.chi2.sf(q, df))}


def acf_bars(ax, r, T, color=st.MainBlue, label='_nolegend_', band=True, mark=None):
    """Bars at lags 1..len(r), the +-1.96/sqrt(T) band, and vertical marks at seasonal lags."""
    lags = np.arange(1, len(r) + 1)
    if mark:
        for m in mark:
            ax.axvline(m, color=st.Orange, lw=0.8, ls=':', label='_nolegend_')
    ax.bar(lags, r, width=0.6, color=color, label=label)
    if band:
        b = 1.96 / np.sqrt(T)
        ax.axhline(b, color=BAND_COL, ls='--', lw=0.9, label=r'$\pm 1.96/\sqrt{T}$')
        ax.axhline(-b, color=BAND_COL, ls='--', lw=0.9, label='_nolegend_')
    ax.axhline(0, color=st.DarkText, lw=0.6)
    ax.set_xlim(0.3, len(r) + 0.7)


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def date_axis(ax, kind='year'):
    """Readable date ticks: 'year' (one tick per year), 'month' (month initials), 'week' (day and month)."""
    if kind == 'year':
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    elif kind == 'month':
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    else:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))


def polymul(a, b):
    return np.convolve(np.asarray(a, float), np.asarray(b, float))


def sarma_process(phi=(), theta=(), Phi=(), Theta=(), s=12):
    """Multiplicative seasonal ARMA phi(L)Phi(L^s) X_t = theta(L)Theta(L^s) e_t as an ArmaProcess (expanded lags)."""
    def seas(c, sign):
        v = np.zeros(s * len(c) + 1)
        v[0] = 1
        for i, x in enumerate(c):
            v[s * (i + 1)] = sign * x
        return v
    ar = polymul(np.r_[1, -np.asarray(phi, float)], seas(Phi, -1))
    ma = polymul(np.r_[1, np.asarray(theta, float)], seas(Theta, 1))
    return ArmaProcess(ar, ma)


def fit_sarima(y, order, sorder, exog=None, trend='n'):
    """Gaussian maximum likelihood of a SARIMA (statsmodels SARIMAX, exact likelihood after differencing)."""
    return SARIMAX(np.asarray(y, float), exog=exog, order=order, seasonal_order=sorder, trend=trend).fit(
        disp=False, maxiter=200)


def aicc(res, n):
    k = len(res.params)
    return float(res.aic + 2 * k * (k + 1) / max(n - k - 1, 1))


def fourier(idx, period, K, t0='2000-01-01'):
    """Fourier terms sin(2 pi k t / period), cos(2 pi k t / period), k = 1..K, with t in days (daily data) or in
    steps (integer index)."""
    if isinstance(idx, pd.DatetimeIndex):
        t = np.asarray((idx - pd.Timestamp(t0)).days, float)
    else:
        t = np.asarray(idx, float)
    cols = {}
    for k in range(1, K + 1):
        cols[f'sin{k}_{period:g}'] = np.sin(2 * np.pi * k * t / period)
        cols[f'cos{k}_{period:g}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=idx if isinstance(idx, pd.DatetimeIndex) else None)


def easter_share(idx, w=10):
    """Easter regressor of a monthly series: the share of the w days before Orthodox Easter Sunday that fall in each
    month (the 'easter[w]' regressor of X-13ARIMA-SEATS, with the Orthodox date)."""
    out = pd.Series(0.0, index=idx)
    for y in sorted(set(idx.year)):
        e = pd.Timestamp(easter(y, EASTER_ORTHODOX))
        days = pd.date_range(e - pd.Timedelta(days=w), e - pd.Timedelta(days=1))
        for m in days:
            key = pd.Timestamp(m.year, m.month, 1)
            if key in out.index:
                out[key] += 1 / w
    return out


def working_days(idx):
    """Number of Monday-Friday days that are not public holidays, in each month."""
    hol = set(ro_holidays(range(idx[0].year, idx[-1].year + 1)).index)
    return pd.Series([sum(1 for d in pd.date_range(m, m + pd.offsets.MonthEnd(0)) if d.dayofweek < 5 and d not in hol)
                      for m in idx], index=idx, dtype=float)


def airline():
    """Box and Jenkins' monthly international airline passengers (thousands), 1949-1960 (R data set via statsmodels)."""
    import statsmodels.api as sm
    d = sm.datasets.get_rdataset('AirPassengers', 'datasets').data
    return pd.Series(d['value'].values.astype(float), index=pd.date_range('1949-01-01', periods=len(d), freq='MS'),
                     name='passengers')


def fig_series(save_it=True):
    """Six Romanian series with seasonality: GDP (NSA), retail trade, industrial production, tourism nights,
    monthly HICP inflation, daily electricity load."""
    gdp = read_eurostat(*GDP_NSA).loc[GDP_START:] / 1000
    ret = read_eurostat(*RETAIL).loc['2010':]
    ipi = read_eurostat(*IPI).loc['2010':]
    tour = read_eurostat(*TOUR).loc['2010':] / 1e6
    infl = 100 * np.log(read_eurostat(*HICP)).diff().loc['2010':]
    load = load_daily()
    panels = [('GDP, not adjusted (bn EUR, 2010 prices)', gdp, st.MainBlue), ('Retail trade volume (2021 = 100)', ret, st.IDAred),
              ('Industrial production (2021 = 100)', ipi, st.Forest), ('Nights in tourist accommodation (millions)', tour, st.Orange),
              ('HICP inflation, month on month (%)', infl, st.Purple), ('Electricity load, daily mean (GW)', load, st.Teal)]
    fig, ax = plt.subplots(3, 2, figsize=(11.0, 6.4))
    for a, (t, s, c) in zip(ax.ravel(), panels):
        a.plot(s.index, s.values, color=c, lw=0.9 if len(s) < 500 else 0.6, label=t)
        a.set_title(t, fontsize=11.5)
        if 'HICP' in t:
            a.axhline(0, color=st.DarkText, lw=0.5)
        a.xaxis.set_major_locator(mdates.YearLocator(base=4 if len(s) < 500 else 1))
        a.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    save('tsa_ch4_series', save_it)
    q = gdp.groupby(gdp.index.quarter).mean()
    mt = tour.loc['2015':'2019']
    mt = mt.groupby(mt.index.month).mean()
    mi = infl.loc['2010':'2019']
    mi = mi.groupby(mi.index.month).mean()
    return {'gdp_first': str(gdp.index[0].date()), 'gdp_last': str(gdp.index[-1].date()), 'gdp_T': int(len(gdp)),
            'gdp_q': q.round(4).tolist(), 'gdp_q4_q1': float(q[4] / q[1] - 1),
            'ret_last': str(ret.index[-1].date()), 'tour_last': str(tour.index[-1].date()),
            'tour_aug_jan': float(mt[8] / mt[1]), 'tour_max_m': int(mt.idxmax()), 'tour_min_m': int(mt.idxmin()),
            'infl_month_mean': mi.round(3).tolist(), 'infl_last': str(infl.index[-1].date()),
            'load_first': str(load.index[0].date()), 'load_last': str(load.index[-1].date()), 'load_T': int(len(load)),
            'load_mean': float(load.mean())}


def fig_seasonal_plot(save_it=True):
    """Seasonal plot of tourism nights (one line per year) and the quarterly subseries of GDP."""
    tour = read_eurostat(*TOUR).loc['2012':'2025'] / 1e6
    gdp = read_eurostat(*GDP_NSA).loc['2005':] / 1000
    fig, ax = plt.subplots(1, 2, figsize=(11.0, 3.9), gridspec_kw={'width_ratios': [1.25, 1]})
    years = sorted(set(tour.index.year))
    cmap = plt.get_cmap('viridis')
    for i, y in enumerate(years):
        s = tour.loc[str(y)]
        special = y in (2020, 2021)
        ax[0].plot(s.index.month, s.values, color=st.IDAred if y == 2020 else cmap(i / (len(years) - 1)),
                   lw=1.8 if special else 1.0, ls='--' if special else '-', label=str(y) if y in (2012, 2020, 2021, 2025) else '_nolegend_')
    ax[0].set_xticks(range(1, 13))
    ax[0].set_xticklabels(list('JFMAMJJASOND'))
    ax[0].set_title('Tourism nights by month, 2012-2025 (millions)')
    st.legend_outside_bottom(ax[0], ncol=4, y=-0.13)
    cols = [st.MainBlue, st.Forest, st.Orange, st.IDAred]
    for q in range(1, 5):
        s = gdp[gdp.index.quarter == q]
        x = (q - 1) + np.linspace(0.1, 0.9, len(s))
        ax[1].plot(x, s.values, color=cols[q - 1], lw=1.1, label=f'Q{q}')
        ax[1].hlines(s.mean(), q - 1 + 0.1, q - 1 + 0.9, color=st.DarkText, lw=1.6, label='quarter mean' if q == 1 else '_nolegend_')
    ax[1].set_xticks([0.5, 1.5, 2.5, 3.5])
    ax[1].set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
    ax[1].set_title('GDP subseries by quarter, 2005-2026 (bn EUR)')
    st.legend_outside_bottom(ax[1], ncol=5, y=-0.13)
    plt.tight_layout()
    save('tsa_ch4_seasonal_plot', save_it)
    t19 = tour.loc['2019']
    return {'tour19_aug': float(t19.iloc[7]), 'tour19_jan': float(t19.iloc[0]), 'tour20_apr': float(tour.loc['2020-04-01']),
            'tour19_apr': float(tour.loc['2019-04-01'])}


def fig_sa_nsa(save_it=True):
    """Eurostat seasonally and calendar adjusted (SCA) and unadjusted (NSA) series: GDP and retail trade; the implied
    seasonal-calendar factors NSA/SCA."""
    g, gs = read_eurostat(*GDP_NSA).loc['2010':] / 1000, read_eurostat(*GDP_SCA).loc['2010':] / 1000
    r, rs = read_eurostat(*RETAIL).loc['2015':], read_eurostat(*RETAIL_SCA).loc['2015':]
    fg, fr = (g / gs).dropna(), (r / rs).dropna()
    fig, ax = plt.subplots(2, 2, figsize=(11.0, 5.4))
    ax[0, 0].plot(g.index, g.values, color=st.MainBlue, lw=1.0, label='not adjusted (NSA)')
    ax[0, 0].plot(gs.index, gs.values, color=st.IDAred, lw=1.6, label='adjusted (SCA)')
    ax[0, 0].set_title('GDP, bn EUR (2010 prices)')
    st.legend_outside_bottom(ax[0, 0], ncol=2, y=-0.14)
    ax[0, 1].plot(r.index, r.values, color=st.MainBlue, lw=0.9, label='not adjusted (NSA)')
    ax[0, 1].plot(rs.index, rs.values, color=st.IDAred, lw=1.6, label='adjusted (SCA)')
    st.legend_outside_bottom(ax[0, 1], ncol=2, y=-0.14)
    ax[0, 1].set_title('Retail trade volume, 2021 = 100')
    cols = [st.MainBlue, st.Forest, st.Orange, st.IDAred]
    for q in range(1, 5):
        s = fg[fg.index.quarter == q]
        ax[1, 0].plot(s.index, s.values, color=cols[q - 1], lw=1.3, marker='o', ms=2.5, label=f'Q{q}')
    ax[1, 0].axhline(1, color=st.DarkText, lw=0.5)
    ax[1, 0].set_title('GDP: factor NSA / SCA by quarter')
    st.legend_outside_bottom(ax[1, 0], ncol=4, y=-0.16)
    ax[1, 1].plot(fr.index, fr.values, color=st.Purple, lw=1.0, label='retail: factor NSA / SCA')
    ax[1, 1].axhline(1, color=st.DarkText, lw=0.5)
    ax[1, 1].set_title('Retail trade: factor NSA / SCA')
    st.legend_outside_bottom(ax[1, 1], ncol=1, y=-0.16)
    plt.tight_layout(h_pad=1.5)
    save('tsa_ch4_sa_nsa', save_it)
    fq = fg.loc['2015':'2019'].groupby(fg.loc['2015':'2019'].index.quarter).mean()
    gr_nsa = 100 * np.log(g).diff().loc['2025':]
    gr_sca = 100 * np.log(gs).diff().loc['2025':]
    fm = fr.loc['2016':'2019']
    fm = fm.groupby(fm.index.month).mean()
    return {'fq': fq.round(4).tolist(), 'qoq_nsa': gr_nsa.round(2).tolist(), 'qoq_sca': gr_sca.round(2).tolist(),
            'qoq_dates': [str(d.date()) for d in gr_nsa.index], 'f_dec': float(fm[12]), 'f_jan': float(fm[1]),
            'sum_nsa_2024': float(g.loc['2024'].sum()), 'sum_sca_2024': float(gs.loc['2024'].sum())}

## Run

In [ ]:
print(fig_series())
print(fig_seasonal_plot())
print(fig_sa_nsa())

![tsa_ch4_series](./tsa_ch4_series.png)

Output: `tsa_ch4_series.pdf`

![tsa_ch4_seasonal_plot](./tsa_ch4_seasonal_plot.png)

Output: `tsa_ch4_seasonal_plot.pdf`

![tsa_ch4_sa_nsa](./tsa_ch4_sa_nsa.png)

Output: `tsa_ch4_sa_nsa.pdf`